In [1]:
import xarray as xr
from dask.distributed import Client, LocalCluster
import os
import glob as glob
import glide.science_data_processing.L1A as L1A

imager = "WFI"
# 1. Set environment variables to prevent underlying library thread conflicts
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# 2. Configure a local cluster optimized for 128 GB RAM and 64 cores
# We restrict to 16 workers with 4 threads each to minimize HDF5 lock contention
cluster = LocalCluster(
    n_workers=16,          # 16 separate Python processes
    threads_per_worker=4,  # 4 threads per process (64 total threads used)
    memory_limit="7.5GB",  # 16 * 7.5GB = 120GB (leaves 8GB safety buffer for OS)
)

# 3. Connect Dask to this custom infrastructure
client = Client(cluster)

# Print the dashboard URL so you can monitor progress live
print(f"Dask Dashboard is live at: {client.dashboard_link}")


file_paths = sorted(glob.glob(f"/data/L1A/CARRUTHERS_GCI-{imager}_L1A-DRK_202601[1-3][0-9]_v1.0.nc"))
print(file_paths)

ds_all = xr.open_mfdataset(
    file_paths, 
    combine="nested",
    engine='netcdf4', 
    concat_dim="time", 
    chunks={"time": 10},  # Lazy loading using Dask (adjust chunk size as needed)
    parallel=True         # Speeds up parsing metadata across cores
)

# 3. Access your L1A wrapper
l1a_all = L1A.L1A(ds_all)

# 3. Now ds_all is a single xarray Dataset with all your data combined along 'time'
# Save dataset
ds_all.to_netcdf(f"products/CARRUTHERS_GCI-{imager}_L1A-DRK_combined_v1.0.nc", mode="w")

/home/jacob/miniconda3/envs/carruthers-sdc/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36159 instead
  warnings.warn(


Dask Dashboard is live at: http://127.0.0.1:36159/status
['/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260110_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260111_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260112_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260113_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260114_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260115_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260116_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260117_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260118_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260119_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260120_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260121_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260122_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260123_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260124_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_L1A-DRK_20260125_v1.0.nc', '/data/L1A/CARRUTHERS_GCI-WFI_

/tmp/ipykernel_702527/1549465671.py:31: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_all = xr.open_mfdataset(
Exception ignored in: <function CachingFileManager.__del__ at 0x7f760a673f60>
Traceback (most recent call last):
  File "/home/jacob/miniconda3/envs/carruthers-sdc/lib/python3.11/site-packages/xarray/backends/file_manager.py", line 258, in __del__
    self.close(needs_lock=False)
  File "/home/jacob/miniconda3/envs/carruthers-sdc/lib/python3.11/site-packages/xarray/backends/file_manager.py", line 242, in close
    file.close()
  File "src/netCDF4/_netCDF4.pyx", line 2680, in netCDF4._netCDF4.Dataset.close
  File "src/netCDF4/_netCDF4.py

RuntimeError: NetCDF: Can't open HDF5 attribute